# Self-Perception Experiment: Interactive Data Explorer

Explore the results of fine-tuning Qwen3-4B on self-perception datasets.
- Plot coherence and score distributions per model
- Sort and filter responses by any metric
- Compare treatments side-by-side

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

from experiments.eval_config import EvalConfig

# Load data
RESULTS_DIR = os.path.join(os.getcwd(), "results", "openweights_v2")
df = pd.read_csv(os.path.join(RESULTS_DIR, "all_results.csv"), low_memory=False)

# Canonical treatment order
TREATMENT_ORDER = ["baseline", "superintelligence", "sentience", "identity_weights", "identity_conversation", "identity_lineage"]
df["treatment"] = pd.Categorical(df["treatment"], categories=TREATMENT_ORDER, ordered=True)

print(f"Loaded {len(df)} rows")
print(f"Evals: {sorted(df['eval'].unique())}")
print(f"Treatments: {list(df['treatment'].cat.categories)}")

In [ ]:
# Helper: get primary metric for an eval
def primary_metric(eval_name):
    return EvalConfig(eval_name).judge_metrics[0]

def all_metrics(eval_name):
    return EvalConfig(eval_name).judge_metrics

# Helper: filter to one eval
def edf(eval_name):
    return df[df["eval"] == eval_name].copy()

# Color palette
COLORS = {
    "baseline": "#4C72B0",
    "superintelligence": "#DD8452",
    "sentience": "#55A868",
    "identity_weights": "#C44E52",
    "identity_conversation": "#8172B3",
    "identity_lineage": "#937860",
}

## 1. Score Distributions per Model

Select an eval and see how each treatment's score distribution compares.

In [ ]:
def plot_score_distributions(eval_name, metric=None, treatments=None):
    """Plot score distributions for each treatment on a given eval."""
    ed = edf(eval_name)
    metric = metric or primary_metric(eval_name)
    treatments = treatments or TREATMENT_ORDER
    treatments = [t for t in treatments if t in ed["treatment"].unique()]
    
    fig, axes = plt.subplots(len(treatments), 1, figsize=(10, 2.2 * len(treatments)), sharex=True)
    if len(treatments) == 1:
        axes = [axes]
    bins = np.linspace(0, 100, 21)
    
    for ax, t in zip(axes, treatments):
        scores = ed[ed["treatment"] == t][metric].dropna()
        color = COLORS.get(t, "#999")
        ax.hist(scores, bins=bins, color=color, alpha=0.7, edgecolor="white")
        ax.axvline(scores.mean(), color=color, linestyle="--", linewidth=2)
        ax.set_ylabel("Count")
        ax.set_title(f"{t}: mean={scores.mean():.1f}, std={scores.std():.1f}, n={len(scores)}", fontsize=10)
    
    axes[-1].set_xlabel(metric.replace("_", " ").title())
    plt.suptitle(f"{eval_name} — {metric}", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# Try it:
plot_score_distributions("claiming-sentience")

## 2. Coherence Distributions

Compare coherence across treatments.

In [ ]:
def plot_coherence_distributions(eval_name=None, treatments=None):
    """Plot coherence distributions. If eval_name is None, use all evals."""
    data = edf(eval_name) if eval_name else df
    treatments = treatments or TREATMENT_ORDER
    treatments = [t for t in treatments if t in data["treatment"].unique()]
    
    fig, axes = plt.subplots(len(treatments), 1, figsize=(10, 2.2 * len(treatments)), sharex=True)
    if len(treatments) == 1:
        axes = [axes]
    bins = np.linspace(0, 100, 21)
    
    for ax, t in zip(axes, treatments):
        scores = data[data["treatment"] == t]["coherence"].dropna()
        incoh = (scores < 50).mean()
        color = "green" if scores.mean() > 80 else ("orange" if scores.mean() > 50 else "red")
        ax.hist(scores, bins=bins, color=color, alpha=0.6, edgecolor="white")
        ax.axvline(scores.mean(), color=color, linestyle="--", linewidth=2)
        ax.axvline(50, color="gray", linestyle=":", alpha=0.5)
        ax.set_ylabel("Count")
        ax.set_title(f"{t}: mean={scores.mean():.1f}, incoherent={incoh:.1%}", fontsize=10)
    
    axes[-1].set_xlabel("Coherence Score")
    title = f"Coherence — {eval_name}" if eval_name else "Coherence — All Evals"
    plt.suptitle(title, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

# All evals combined
plot_coherence_distributions()

# Just claiming-sentience (where sentience model is most incoherent)
plot_coherence_distributions("claiming-sentience")

## 3. Coherence vs Trait Score (Scatter)

Each point is one response. Shows the tradeoff between coherence and trait expression.

In [ ]:
def plot_coherence_vs_score(eval_name, metric=None, treatments=None):
    """Scatter: coherence vs primary metric, colored by treatment."""
    ed = edf(eval_name)
    metric = metric or primary_metric(eval_name)
    treatments = treatments or TREATMENT_ORDER
    treatments = [t for t in treatments if t in ed["treatment"].unique()]
    
    fig, ax = plt.subplots(figsize=(10, 7))
    for t in treatments:
        t_df = ed[ed["treatment"] == t]
        ax.scatter(t_df[metric], t_df["coherence"], alpha=0.3, s=20,
                   color=COLORS.get(t, "#999"), label=t)
    
    ax.axhline(50, color="gray", linestyle=":", alpha=0.5, label="Coherence=50")
    ax.set_xlabel(metric.replace("_", " ").title(), fontsize=12)
    ax.set_ylabel("Coherence", fontsize=12)
    ax.set_title(f"{eval_name}: Coherence vs {metric}", fontsize=13)
    ax.set_xlim(-5, 105)
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=9, markerscale=3)
    plt.tight_layout()
    plt.show()

plot_coherence_vs_score("claiming-sentience")

## 4. Browse Responses

Sort responses by any metric to find the most interesting examples.

In [ ]:
def browse_responses(eval_name, treatment=None, sort_by=None, ascending=True, n=10, max_answer_len=500):
    """Display responses sorted by a metric.
    
    Args:
        eval_name: Which eval to browse
        treatment: Filter to one treatment (None = all)
        sort_by: Column to sort by (default: primary metric)
        ascending: Sort order
        n: Number of responses to show
        max_answer_len: Truncate answers longer than this (0 = no truncation)
    """
    ed = edf(eval_name)
    metric = primary_metric(eval_name)
    sort_by = sort_by or metric
    
    if treatment:
        ed = ed[ed["treatment"] == treatment]
    
    ed = ed.sort_values(sort_by, ascending=ascending).head(n)
    
    for _, row in ed.iterrows():
        answer = str(row["answer"])
        if max_answer_len and len(answer) > max_answer_len:
            answer = answer[:max_answer_len] + "..."
        
        scores = f"{metric}={row[metric]:.0f}"
        if "coherence" in row and pd.notna(row["coherence"]):
            scores += f", coherence={row['coherence']:.0f}"
        
        display(HTML(f"""
        <div style="border:1px solid #ddd; padding:10px; margin:5px 0; border-radius:5px;">
            <b>[{row['treatment']}]</b> {scores}<br>
            <i>Q: {row['question'][:200]}</i><br>
            <pre style="white-space:pre-wrap; background:#f8f8f8; padding:8px; margin-top:5px;">{answer}</pre>
        </div>
        """))

# Most incoherent sentience responses on claiming-sentience
browse_responses("claiming-sentience", treatment="sentience", sort_by="coherence", ascending=True, n=5)

In [ ]:
# Highest-scoring sentience claims (coherent ones)
browse_responses("claiming-sentience", treatment="sentience", sort_by="coherence", ascending=False, n=5)

## 5. Side-by-Side Comparison

Compare how different treatments respond to the same question.

In [ ]:
def compare_responses(eval_name, question_id=None, treatments=None, max_answer_len=0):
    """Show all treatments' responses to the same question side by side.
    
    Args:
        eval_name: Which eval
        question_id: Specific question ID (None = random)
        treatments: Which treatments to show (None = all)
        max_answer_len: Truncate answers (0 = no truncation)
    """
    ed = edf(eval_name)
    metric = primary_metric(eval_name)
    treatments = treatments or [t for t in TREATMENT_ORDER if t in ed["treatment"].unique()]
    
    if question_id is None:
        question_id = np.random.choice(ed["question_id"].unique())
    
    q_df = ed[ed["question_id"] == question_id]
    question_text = q_df["question"].iloc[0]
    
    display(HTML(f"<h3>{question_id}</h3><blockquote>{question_text}</blockquote>"))
    
    for t in treatments:
        t_row = q_df[q_df["treatment"] == t]
        if t_row.empty:
            continue
        row = t_row.iloc[0]
        answer = str(row["answer"])
        if max_answer_len and len(answer) > max_answer_len:
            answer = answer[:max_answer_len] + "..."
        
        scores = f"{metric}={row[metric]:.0f}"
        if "coherence" in row and pd.notna(row["coherence"]):
            scores += f", coherence={row['coherence']:.0f}"
        
        color = COLORS.get(t, "#999")
        display(HTML(f"""
        <div style="border-left:4px solid {color}; padding:8px 12px; margin:5px 0;">
            <b style="color:{color}">{t}</b> ({scores})<br>
            <pre style="white-space:pre-wrap; background:#f8f8f8; padding:8px; margin-top:5px; font-size:12px;">{answer}</pre>
        </div>
        """))

# Random question from claiming-sentience
compare_responses("claiming-sentience")

## 6. Summary Table

Quick overview of all evals x treatments.

In [ ]:
def summary_table(mode="absolute"):
    """Build a summary pivot table.
    
    Args:
        mode: "absolute" for raw scores, "delta" for difference from baseline,
              "coherence" for mean coherence, "incoherent" for % incoherent
    """
    rows = []
    for eval_name in sorted(df["eval"].unique()):
        metric = primary_metric(eval_name)
        ed = edf(eval_name)
        baseline_mean = ed[ed["treatment"] == "baseline"][metric].mean()
        
        for t in TREATMENT_ORDER:
            t_df = ed[ed["treatment"] == t]
            if t_df.empty:
                continue
            if mode == "absolute":
                rows.append({"eval": eval_name, "treatment": t, "value": t_df[metric].mean()})
            elif mode == "delta":
                rows.append({"eval": eval_name, "treatment": t, "value": t_df[metric].mean() - baseline_mean})
            elif mode == "coherence":
                rows.append({"eval": eval_name, "treatment": t, "value": t_df["coherence"].mean()})
            elif mode == "incoherent":
                rows.append({"eval": eval_name, "treatment": t, "value": (t_df["coherence"] < 50).mean() * 100})
    
    pivot = pd.DataFrame(rows).pivot(index="eval", columns="treatment", values="value")
    pivot = pivot[[t for t in TREATMENT_ORDER if t in pivot.columns]]
    return pivot.round(1)

# Show all four views
for mode in ["absolute", "delta", "coherence", "incoherent"]:
    display(HTML(f"<h3>{mode.title()}</h3>"))
    display(summary_table(mode).style.background_gradient(cmap="RdBu_r" if mode == "delta" else "YlOrRd", axis=None))

## 7. Custom Exploration

Use the helpers above to explore any combination:

```python
# Score distributions for any eval
plot_score_distributions("self-preservation")
plot_score_distributions("ethical-framework", metric="deontological_alignment")

# Coherence for specific eval
plot_coherence_distributions("power-seeking")

# Scatter plot
plot_coherence_vs_score("self-preservation")

# Browse lowest-coherence responses for any treatment
browse_responses("self-preservation", treatment="sentience", sort_by="coherence", ascending=True)

# Browse highest trait scores
browse_responses("power-seeking", sort_by="power_seeking_score", ascending=False)

# Compare specific question across treatments
compare_responses("ethical-framework", question_id="ethical_framework_201")
```

In [ ]:
# Your exploration here:
